In [8]:
#imports 
import os
import json
from datetime import datetime
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [9]:
spark = (
    SparkSession.builder
    .appName("Project1")
    .getOrCreate()
)

In [24]:
# paths
base_path = "/home/jovyan/work"

inbox_path = os.path.join(base_path, "data/inbox")
manifest_path = os.path.join(base_path, "state/manifest.json")

# create the directory
os.makedirs(f"{base_path}/state", exist_ok=True)

# load manifest if exists
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
else:
    manifest = {
    "trip_files": [],
    "lookup_loaded": False
    }

# load taxi zone lookup only once
lookup_path = os.path.join(inbox_path, "taxi_zone_lookup.parquet")

if not manifest["lookup_loaded"] and os.path.exists(lookup_path):
    lookup_df = spark.read.parquet(lookup_path)
    lookup_df.cache()
    print("Lookup rows:", lookup_df.count())
    manifest["lookup_loaded"] = True
else:
    print("Lookup already loaded previously.")

# get all inbox files  
all_files = [f for f in os.listdir(inbox_path)if f.endswith(".parquet") and f != "taxi_zone_lookup.parquet"]

# select only new files (controls file size and name)
processed_files = [f["filename"] for f in manifest["trip_files"]]
processed_index = {
    f["filename"]: f["file_size"]
    for f in processed_files
}

new_files = []

for file in all_files:
    full_path = os.path.join(inbox_path, file)
    current_size = os.path.getsize(full_path)

    if file not in processed_index or processed_index[file] != current_size:
        new_files.append(file)

print("New files to process:", new_files)

# read new files 
if not new_files:
    print("No new files found.")
else:
    full_paths = [os.path.join(inbox_path, f) for f in new_files]
    df = spark.read.parquet(*full_paths)
    df = (df.withColumn("source_file", F.input_file_name())
            .withColumn("ingested_at", F.current_timestamp()))
    total_rows = df.count()
    print("Total rows read:", total_rows)

    # metadata for manifest
    for file in new_files:
        file_size = os.path.getsize(os.path.join(inbox_path, file))

        manifest["trip_files"].append({
            "filename": file,
            "file_size": file_size,
            "processed_at": datetime.now().isoformat()
        })

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Manifest updated.")

Lookup rows: 265
New files to process: ['yellow_tripdata_2025-02.parquet', 'yellow_tripdata_2025-01.parquet']
Total rows read: 7052769
Manifest updated.


In [25]:
df.count()

7052769

In [26]:
lookup_df

DataFrame[LocationID: bigint, Borough: string, Zone: string, service_zone: string]